# 03 — Baseline Cross-Dataset (XGBoost Tunggal)

**Tujuan:** mengukur generalisasi lintas-jaringan pada fitur irisan hasil Semantic Feature Mapping (T2). Protokol: latih pada satu dataset, uji pada dataset lain (dua arah), lalu bandingkan dengan baseline *same-dataset* untuk mengkuantifikasi **generalization gap**.

**Yang diuji:**
- **Model A** (9 fitur irisan kuat): `dur, spkts, dpkts, sbytes, dbytes, smean, dmean, sload, dload`
- **Model B** (11 fitur): Model A + `sinpkt, dinpkt`

**Preprocessing:** z-score (StandardScaler) **per dataset terpisah** (Opsi 3) — perbedaan satuan/skala antar-extractor ternormalisasi tanpa mengarang konversi. Fitur `swin`/`dwin` **dibuang** (mismatch definisi, lihat dokumentasi 10.7).

**Label:** biner attack (1) vs normal (0) untuk tahap awal.

**Metrik utama:** MCC (konsisten Paper 1, tahan *class imbalance*) + F1, akurasi, ROC-AUC.

**Konfigurasi XGBoost (mengikuti Paper 1):** `max_depth=8, learning_rate=0.1, n_estimators=200, subsample=0.8, colsample_bytree=0.8`. Untuk biner: `objective='binary:logistic'`.

> **Kejujuran data:** seluruh angka di notebook ini berasal dari eksekusi nyata di SageMaker atas dataset asli. Tidak ada angka ilustratif.

> Jalankan di SageMaker (butuh `cleaned_100.pkl` untuk CIC + CSV UNSW di `../data/`).

In [ ]:
# --- Bootstrap dependency (SageMaker mereset pip saat stop/start) ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('numpy','numpy'), ('scikit-learn','sklearn'), ('xgboost','xgboost')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pickle, os, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (matthews_corrcoef, f1_score, accuracy_score,
                             roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier

CIC_PKL  = '../../CICDDoS2018/data/cleaned_100.pkl'
# CATATAN: nama berkas UNSW TERTUKAR dengan isinya.
#   UNSW_NB15_testing-set.csv  = 175.341 record  -> dipakai sbg TRAIN
#   UNSW_NB15_training-set.csv =  82.332 record  -> dipakai sbg TEST
UNSW_TRAIN = '../data/UNSW_NB15_testing-set.csv'   # 175.341
UNSW_TEST  = '../data/UNSW_NB15_training-set.csv'  #  82.332
OUT_JSON   = '../cross_dataset_baseline.json'

print('CIC pkl    :', os.path.exists(CIC_PKL))
print('UNSW train :', os.path.exists(UNSW_TRAIN))
print('UNSW test  :', os.path.exists(UNSW_TEST))

In [ ]:
# --- Mapping fitur final (nama kanonik bersama <- CIC <- UNSW) ---
# canonical : (nama_kolom_CIC, nama_kolom_UNSW)
MAP_A = {
    'duration'  : ('Flow Duration',    'dur'),
    'fwd_pkts'  : ('Tot Fwd Pkts',     'spkts'),
    'bwd_pkts'  : ('Tot Bwd Pkts',     'dpkts'),
    'fwd_bytes' : ('TotLen Fwd Pkts',  'sbytes'),
    'bwd_bytes' : ('TotLen Bwd Pkts',  'dbytes'),
    'fwd_mean'  : ('Fwd Pkt Len Mean', 'smean'),
    'bwd_mean'  : ('Bwd Pkt Len Mean', 'dmean'),
    'src_load'  : ('Flow Byts/s',      'sload'),
    'dst_load'  : ('Bwd Pkts/s',       'dload'),
}
MAP_B = dict(MAP_A)
MAP_B.update({
    'fwd_iat'   : ('Fwd IAT Mean',     'sinpkt'),
    'bwd_iat'   : ('Bwd IAT Mean',     'dinpkt'),
})
print('Model A:', list(MAP_A.keys()))
print('Model B:', list(MAP_B.keys()))

In [ ]:
# --- Muat CIC-IDS2018 (un-scale ke skala asli) + label biner ---
with open(CIC_PKL, 'rb') as f:
    d = pickle.load(f)
print('key pkl:', list(d.keys()))

cic_feats = list(d['feature_names'])
X = np.asarray(d['X'], dtype=float)
scaler = d.get('scaler', None)
if scaler is not None and hasattr(scaler, 'scale_') and hasattr(scaler, 'mean_'):
    X_orig = X * scaler.scale_ + scaler.mean_
    print('CIC: un-scaled via scaler.mean_/scale_')
else:
    X_orig = X
    print('CIC: scaler tak tersedia -> pakai X apa adanya')
cic_df = pd.DataFrame(X_orig, columns=cic_feats)

# --- Ambil label CIC biner ---
# Coba beberapa kemungkinan penyimpanan label di pkl.
y_cic = None
for k in ['y', 'labels', 'label', 'Label', 'y_bin', 'target']:
    if k in d:
        y_cic = np.asarray(d[k]); print(f'label CIC diambil dari key: {k!r}'); break
if y_cic is None:
    raise KeyError(f'Label CIC tidak ditemukan di pkl. Key tersedia: {list(d.keys())}. '
                   'Sesuaikan nama key label pada sel ini.')

# Normalisasi ke biner 0/1: 'Benign'/0 -> 0 (normal), selain itu -> 1 (attack).
def to_binary_cic(y):
    y = pd.Series(y).astype(str).str.strip()
    # jika sudah numerik 0/1
    uniq = set(y.unique())
    if uniq <= {'0','1','0.0','1.0'}:
        return y.astype(float).astype(int).values
    is_benign = y.str.lower().isin(['benign','normal','0','0.0'])
    return (~is_benign).astype(int).values

y_cic_bin = to_binary_cic(y_cic)
print('CIC shape:', cic_df.shape, '| distribusi label biner:', dict(pd.Series(y_cic_bin).value_counts()))

In [ ]:
# --- Muat UNSW-NB15 (train=175k, test=82k) + label biner ---
unsw_tr = pd.read_csv(UNSW_TRAIN)
unsw_te = pd.read_csv(UNSW_TEST)
print('UNSW train:', unsw_tr.shape, '| test:', unsw_te.shape)

# label biner UNSW: kolom 'label' (0 normal / 1 attack)
assert 'label' in unsw_tr.columns, f'kolom label UNSW tak ada: {list(unsw_tr.columns)[:10]}...'
y_unsw_tr = unsw_tr['label'].astype(int).values
y_unsw_te = unsw_te['label'].astype(int).values
print('UNSW train label:', dict(pd.Series(y_unsw_tr).value_counts()))
print('UNSW test  label:', dict(pd.Series(y_unsw_te).value_counts()))

In [ ]:
# --- Util: bangun matriks fitur kanonik dari sebuah dataset ---
def build_matrix(df, mapping, side):
    """side='cic' -> ambil kolom CIC; side='unsw' -> ambil kolom UNSW. Rename ke nama kanonik."""
    idx = 0 if side == 'cic' else 1
    cols, canon = [], []
    for c, pair in mapping.items():
        col = pair[idx]
        if col not in df.columns:
            raise KeyError(f"[{side}] kolom '{col}' (kanonik '{c}') tak ditemukan")
        cols.append(col); canon.append(c)
    out = df[cols].copy()
    out.columns = canon
    # bersihkan inf/nan
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float)

def zscore_fit_transform(train_df):
    sc = StandardScaler().fit(train_df.values)
    return sc

def evaluate(y_true, y_pred, y_prob=None):
    m = dict(
        mcc = float(matthews_corrcoef(y_true, y_pred)),
        f1  = float(f1_score(y_true, y_pred, zero_division=0)),
        acc = float(accuracy_score(y_true, y_pred)),
    )
    try:
        if y_prob is not None and len(np.unique(y_true)) > 1:
            m['roc_auc'] = float(roc_auc_score(y_true, y_prob))
    except Exception:
        pass
    m['confusion'] = confusion_matrix(y_true, y_pred).tolist()
    return m

def make_xgb():
    return XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        max_depth=8, learning_rate=0.1, n_estimators=200,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=42, tree_method='hist')

In [ ]:
# --- Protokol lengkap untuk satu set fitur (Model A atau B) ---
# Skenario:
#   same_cic  : latih CIC (split internal)  -> uji CIC   (baseline atas)
#   same_unsw : latih UNSW(train) -> uji UNSW(test)      (baseline atas)
#   cic2unsw  : latih CIC -> uji UNSW(test)              (cross)
#   unsw2cic  : latih UNSW(train) -> uji CIC             (cross)
# generalization gap = same-dataset MCC - cross MCC (arah yg sesuai sumber latih)
from sklearn.model_selection import train_test_split

def run_model(mapping, name):
    print('='*70); print(f'MODEL {name}  ({len(mapping)} fitur: {list(mapping.keys())})'); print('='*70)

    # matriks fitur kanonik
    Xc = build_matrix(cic_df, mapping, 'cic')
    Xu_tr = build_matrix(unsw_tr, mapping, 'unsw')
    Xu_te = build_matrix(unsw_te, mapping, 'unsw')

    # z-score PER dataset (fit di data latih masing-masing)
    # CIC: split internal untuk baseline same-dataset
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
        Xc.values, y_cic_bin, test_size=0.3, random_state=42, stratify=y_cic_bin)
    sc_cic = StandardScaler().fit(Xc_tr)
    Xc_tr_s, Xc_te_s = sc_cic.transform(Xc_tr), sc_cic.transform(Xc_te)
    Xc_full_s = sc_cic.transform(Xc.values)   # seluruh CIC utk dipakai uji cross

    sc_unsw = StandardScaler().fit(Xu_tr.values)
    Xu_tr_s = sc_unsw.transform(Xu_tr.values)
    Xu_te_s = sc_unsw.transform(Xu_te.values)

    res = {}

    # --- baseline same-dataset: CIC ---
    clf = make_xgb(); clf.fit(Xc_tr_s, yc_tr)
    p = clf.predict(Xc_te_s); pr = clf.predict_proba(Xc_te_s)[:,1]
    res['same_cic'] = evaluate(yc_te, p, pr)
    clf_cic_full = clf  # model dilatih di CIC (subset latih) utk cross ke UNSW

    # --- baseline same-dataset: UNSW ---
    clf = make_xgb(); clf.fit(Xu_tr_s, y_unsw_tr)
    p = clf.predict(Xu_te_s); pr = clf.predict_proba(Xu_te_s)[:,1]
    res['same_unsw'] = evaluate(y_unsw_te, p, pr)
    clf_unsw = clf  # model dilatih di UNSW utk cross ke CIC

    # --- cross: latih CIC -> uji UNSW(test) ---
    p = clf_cic_full.predict(Xu_te_s); pr = clf_cic_full.predict_proba(Xu_te_s)[:,1]
    res['cic2unsw'] = evaluate(y_unsw_te, p, pr)

    # --- cross: latih UNSW -> uji CIC(seluruh) ---
    p = clf_unsw.predict(Xc_full_s); pr = clf_unsw.predict_proba(Xc_full_s)[:,1]
    res['unsw2cic'] = evaluate(y_cic_bin, p, pr)

    # --- generalization gap (MCC) ---
    res['gap_cic_train']  = round(res['same_cic']['mcc']  - res['cic2unsw']['mcc'], 4)
    res['gap_unsw_train'] = round(res['same_unsw']['mcc'] - res['unsw2cic']['mcc'], 4)

    # ringkas cetak
    for k in ['same_cic','same_unsw','cic2unsw','unsw2cic']:
        m = res[k]
        print(f"  {k:10s}  MCC={m['mcc']:.4f}  F1={m['f1']:.4f}  ACC={m['acc']:.4f}"
              f"  AUC={m.get('roc_auc', float('nan')):.4f}")
    print(f"  gap (latih CIC)  = {res['gap_cic_train']}")
    print(f"  gap (latih UNSW) = {res['gap_unsw_train']}")
    return res

In [ ]:
# --- Jalankan Model A & Model B ---
results = {}
results['model_A'] = run_model(MAP_A, 'A')
results['model_B'] = run_model(MAP_B, 'B')

In [ ]:
# --- Ringkasan perbandingan A vs B + simpan hasil ---
def row(tag, r):
    return dict(model=tag,
                same_cic_mcc=round(r['same_cic']['mcc'],4),
                same_unsw_mcc=round(r['same_unsw']['mcc'],4),
                cic2unsw_mcc=round(r['cic2unsw']['mcc'],4),
                unsw2cic_mcc=round(r['unsw2cic']['mcc'],4),
                gap_cic=r['gap_cic_train'], gap_unsw=r['gap_unsw_train'])

summary = pd.DataFrame([row('A (9 fitur)', results['model_A']),
                        row('B (11 fitur)', results['model_B'])])
print(summary.to_string(index=False))

meta = dict(
    deskripsi='Baseline cross-dataset XGBoost tunggal (biner attack/normal), z-score per dataset.',
    config=dict(max_depth=8, learning_rate=0.1, n_estimators=200,
                subsample=0.8, colsample_bytree=0.8, objective='binary:logistic'),
    model_A_features=list(MAP_A.keys()),
    model_B_features=list(MAP_B.keys()),
    results=results,
)
with open(OUT_JSON, 'w') as f:
    json.dump(meta, f, indent=2)
print('\nSaved:', OUT_JSON)
print('\nCatatan interpretasi: gap besar (same-dataset MCC tinggi, cross MCC rendah) = bukti')
print('ketidakmampuan generalisasi lintas-jaringan (gap #1) -> motivasi Semantic Feature Mapping + adversarial.')